In [1]:
from langchain_core.documents import Document

In [2]:
# sample_doc=Document(
#     page_content="Hello World!",
#     metadata={"source":"https://www.google.com"}
# )

In [5]:
# sample_doc

In [4]:
# type(sample_doc)

In [5]:
# # Text data
# from langchain_community.document_loaders.text import TextLoader

# loader=TextLoader("data/Python.txt",encoding="utf-8")

In [6]:
# document=loader.load()

In [1]:
# document

In [8]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader=PyPDFLoader("data/research2.pdf")

In [9]:
# document=pdf_loader.load()

In [2]:
# document

In [11]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader=PyMuPDFLoader("data/research2.pdf")

In [12]:
# document=pdf_loader.load()

In [3]:
# document

# Ingestion Pipeline

In [14]:
import os

In [15]:
def load_all_pdfs():
    folder_path="data"
    num_docs=0
    all_docs=[]
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path=os.path.join(folder_path,filename)
            loader=PyPDFLoader(pdf_path)
            doc=loader.load()

            all_docs.extend(doc)
            num_docs+=1

    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs

In [16]:
all_pdf_documents=load_all_pdfs()

total pdfs: 1
total pages: 21


In [17]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

# Chunks

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [19]:
chunks=split_docs(all_pdf_documents)

In [20]:
len(chunks)

244

In [21]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-28T00:54:45+00:00', 'author': '', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\research2.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}, page_content='1\nRetrieval-Augmented Generation for Large\nLanguage Models: A Survey\nYunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng\nWangc, and Haofen Wang a,c\naShanghai Research Institute for Intelligent Autonomous Systems, Tongji University\nbShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University\ncCollege of Design and Innovation, Tongji University\nAbstract—Large Language Models (LLMs) showcase impres-'),
 Document(metadata={'producer': 'pdfTeX-1.40.25', '

# Embeddings

In [22]:
from sentence_transformers import SentenceTransformer

In [23]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading model......",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embeddings shape:",embeddings.shape)
        return embeddings

In [24]:
embedding_manager=EmbeddingManager()

loading model...... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


# Vector Store

In [25]:
import chromadb
import uuid

In [26]:
class VectorStoreManager:
    def __init__(self,persist_dir="data/vector_store",collection_name="pdf_documents"):
        self.collection_name=collection_name
        self.persist_dir=persist_dir
        self.collection=None
        self.client=None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_dir,exist_ok=True)
        # create a client
        self.client=chromadb.PersistentClient(path=self.persist_dir)

        # create the collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection=",self.collection_name)
        print("docs in collection:",self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings does not match")
    
        # store => ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
    
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
    
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
    
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
    
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
    
        print("Total documents added in vector store:", len(documents_content))
        print("Docs in collection:", self.collection.count())

In [27]:
vector_store=VectorStoreManager()

initialized the vector store with collection= pdf_documents
docs in collection: 244


In [28]:
# data => documents => chunks => embeddings => store in vector store

texts=[doc.page_content for doc in chunks]
embedding=embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
Total documents added in vector store: 244
Docs in collection: 488


# Retrieval

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # Convert query → embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):
                similarity_score = 1 - distance  # cosine similarity

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")
        else:
            print("No document found")

        return retrieved_docs  

In [31]:
rag_retriever=RAGRetriever(embedding_manager,vector_store)

In [32]:
rag_retriever.retrieve("what is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_b3715225-8224-42e2-9f01-2229fd7fce8c',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'title': '',
   'subject': '',
   'source': 'data\\research2.pdf',
   'page_label': '1',
   'creator': 'LaTeX with hyperref',
   'content_length': 288,
   'producer': 'pdfTeX-1.40.25',
   'author': '',
   'keywords': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'page': 0,
   'doc_index': 11,
   'total_pages': 21,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'trapped': '/False'},
  'distance': 0.46291476488113403,
  'similarity_score': 0.537085235118866,
  'rank': 1},
 {'id': 'doc_f628021a-e6ba-

# Integerate with LLM

In [38]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    groq_api_key=API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

In [39]:
def generate_output(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt=f"""use given context to generate the answer for the query 
                Context:{context} 
                Query: {query}"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [40]:
answer=generate_output("what is RAG",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 3 documents


In [41]:
answer

'Based on the given context, RAG stands for "Reinforcement Learning of Augmented Generators" or more specifically, "Reinforced Augmented Generator" or "Reinforced Augmented Graph" but most likely "Reinforced Augmented Generator" is not correct as it is not a common term in the field of computer science. \n\nHowever, based on the context, it seems that RAG is more likely to be "Reinforced Augmented Graph" or "Reinforced Augmented Generator" is not correct but "Reinforced Augmented Graph" is not a common term either. \n\nBut based on the context, it seems that RAG is more likely to be "Reinforced Augmented Generator" is not correct but "Reinforced Augmented Graph" is not a common term either. \n\nHowever, based on the context, it seems that RAG is more likely to be "Reinforced Augmented Graph" is not a common term but "Reinforced Augmented Generator" is not correct but "Reinforced Augmented Graph" is not a common term either. \n\nHowever, based on the context, it seems that RAG is more l